# 3. Modelagem com LightGBM

## Objetivo

Treinar e avaliar um modelo de predição de óbito hospitalar por COVID-19 usando **LightGBM** (Light Gradient Boosting Machine).

## O que é LightGBM?

**LightGBM** é um algoritmo de aprendizado de máquina baseado em árvores de decisão que:
- Constrói árvores sequencialmente, onde cada árvore corrige erros da anterior
- É muito rápido e eficiente em memória
- Funciona bem com dados desbalanceados
- Fornece importância das features automaticamente

### Analogia: Aprendendo com erros
Imagine que você está aprendendo a prever se um paciente vai sobreviver:
1. **Primeira árvore**: Faz uma previsão inicial (pode errar muito)
2. **Segunda árvore**: Aprende com os erros da primeira
3. **Terceira árvore**: Aprende com os erros da segunda
4. E assim por diante...

No final, a previsão é a soma de todas as árvores, resultando em uma previsão muito melhor!

---

## Seção 1: Importações e Carregamento de Dados

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json

# LightGBM
import lightgbm as lgb

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    auc
)

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✓ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar dados processados
processed_dir = Path("../data/processed")

print("Carregando dados processados...")
X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").squeeze()
y_test = pd.read_csv(processed_dir / "y_test.csv").squeeze()

print(f"✓ Dados carregados com sucesso!")
print(f"\nConjunto de treino: {X_train.shape}")
print(f"Conjunto de teste: {X_test.shape}")
print(f"\nVariáveis: {X_train.columns.tolist()}")

## Seção 2: Treinamento do Modelo LightGBM

### Hiperparâmetros
Hiperparâmetros são configurações que controlam como o modelo aprende. Alguns importantes:

- **num_leaves**: Número máximo de folhas em cada árvore (complexidade)
- **learning_rate**: Quão rápido o modelo aprende (0-1)
- **n_estimators**: Número de árvores a construir
- **max_depth**: Profundidade máxima de cada árvore
- **min_child_samples**: Mínimo de amostras em uma folha

In [ ]:
print("="*80)
print("TREINAMENTO DO MODELO LIGHTGBM")
print("="*80)

# Definir hiperparâmetros
params = {
    'objective': 'binary',  # Classificação binária (óbito ou não)
    'metric': 'auc',  # Métrica de avaliação
    'num_leaves': 31,  # Número de folhas
    'learning_rate': 0.05,  # Taxa de aprendizado
    'feature_fraction': 0.8,  # Usar 80% das features
    'bagging_fraction': 0.8,  # Usar 80% das amostras
    'bagging_freq': 5,  # Fazer bagging a cada 5 iterações
    'verbose': -1,  # Sem mensagens de progresso
    'is_unbalance': True,  # Dados desbalanceados
}

print("\nHiperparâmetros:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Criar dataset LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

print("\n✓ Datasets LightGBM criados")

# Treinar modelo
print("\nTreinando modelo...")
model_lgb = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[valid_data],
    callbacks=[
        lgb.log_evaluation(period=20),
        lgb.early_stopping(stopping_rounds=10)
    ]
)

print("\n✓ Modelo treinado com sucesso!")

## Seção 3: Avaliação do Modelo

### Métricas de Avaliação

Para um problema de classificação desbalanceado, usamos várias métricas:

- **Acurácia**: Percentual de previsões corretas (pode enganar com dados desbalanceados)
- **Precisão**: De todos os que o modelo previu como óbito, quantos realmente morreram?
- **Recall (Sensibilidade)**: De todos os que realmente morreram, quantos o modelo identificou?
- **F1-Score**: Média harmônica entre precisão e recall
- **AUC-ROC**: Área sob a curva ROC (melhor métrica para desbalanceamento)

In [ ]:
# Fazer previsões
print("="*80)
print("AVALIAÇÃO DO MODELO LIGHTGBM")
print("="*80)

# Previsões de probabilidade
y_pred_proba_train = model_lgb.predict(X_train)
y_pred_proba_test = model_lgb.predict(X_test)

# Previsões binárias (threshold = 0.5)
y_pred_train = (y_pred_proba_train >= 0.5).astype(int)
y_pred_test = (y_pred_proba_test >= 0.5).astype(int)

print("\nPrevisões realizadas!")
print(f"Distribuição de previsões no teste:")
print(f"  Sobreviveu (0): {(y_pred_test == 0).sum()}")
print(f"  Óbito (1): {(y_pred_test == 1).sum()}")

In [ ]:
# Calcular métricas
metrics_train = {
    'Acurácia': accuracy_score(y_train, y_pred_train),
    'Precisão': precision_score(y_train, y_pred_train, zero_division=0),
    'Recall': recall_score(y_train, y_pred_train, zero_division=0),
    'F1-Score': f1_score(y_train, y_pred_train, zero_division=0),
    'AUC-ROC': roc_auc_score(y_train, y_pred_proba_train)
}

metrics_test = {
    'Acurácia': accuracy_score(y_test, y_pred_test),
    'Precisão': precision_score(y_test, y_pred_test, zero_division=0),
    'Recall': recall_score(y_test, y_pred_test, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_test, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba_test)
}

# Exibir métricas
print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TREINO:")
for metric, value in metrics_train.items():
    print(f"  {metric}: {value:.4f}")

print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TESTE:")
for metric, value in metrics_test.items():
    print(f"  {metric}: {value:.4f}")

# Comparação
print("\nCOMPARAÇÃO TREINO vs TESTE:")
for metric in metrics_train.keys():
    diff = metrics_train[metric] - metrics_test[metric]
    print(f"  {metric}: Treino={metrics_train[metric]:.4f}, Teste={metrics_test[metric]:.4f}, Diferença={diff:.4f}")

print("\n⚠ INTERPRETAÇÃO:")
print("  - Diferença pequena (< 0.05): Modelo bem generalizado")
print("  - Diferença grande (> 0.1): Possível overfitting")

In [ ]:
# Matriz de confusão
print("\nMATRIZ DE CONFUSÃO - TESTE:")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

print("\nInterpretação:")
print(f"  Verdadeiros Negativos (TN): {cm[0, 0]} - Corretamente previu sobrevivência")
print(f"  Falsos Positivos (FP): {cm[0, 1]} - Previu óbito mas paciente sobreviveu")
print(f"  Falsos Negativos (FN): {cm[1, 0]} - Previu sobrevivência mas paciente morreu")
print(f"  Verdadeiros Positivos (TP): {cm[1, 1]} - Corretamente previu óbito")

# Visualizar matriz de confusão
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Sobreviveu', 'Óbito'],
            yticklabels=['Sobreviveu', 'Óbito'])
ax.set_xlabel('Previsão')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão - LightGBM (Teste)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_lightgbm_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classificador Aleatório')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curva ROC - LightGBM (Teste)', fontsize=12, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/11_lightgbm_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")
print("\n✓ Gráfico salvo")

## Seção 4: Importância das Features

### O que é importância de feature?
Importância de feature mede o quanto cada variável contribui para as previsões do modelo. Variáveis importantes:
- Aparecem frequentemente nas árvores
- Fazem divisões que reduzem muito o erro
- Têm grande impacto nas previsões

In [ ]:
# Importância das features
print("="*80)
print("IMPORTÂNCIA DAS FEATURES - LIGHTGBM")
print("="*80)

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importância': model_lgb.feature_importance(importance_type='gain')
}).sort_values('Importância', ascending=False)

print("\nTop 15 Features mais importantes:")
print(feature_importance.head(15).to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['Importância'], color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Importância (Gain)')
ax.set_title('Top 15 Features mais Importantes - LightGBM', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/12_lightgbm_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

## Seção 5: Salvando o Modelo

In [ ]:
# Salvar modelo
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

# Salvar modelo LightGBM
model_lgb.save_model(str(results_dir / "lightgbm_model.txt"))

# Salvar métricas
metrics_summary = {
    'algorithm': 'LightGBM',
    'train_metrics': metrics_train,
    'test_metrics': metrics_test,
    'feature_importance': feature_importance.to_dict('list')
}

with open(results_dir / "lightgbm_metrics.json", 'w') as f:
    json.dump(metrics_summary, f, indent=4)

# Salvar feature importance
feature_importance.to_csv(results_dir / "lightgbm_feature_importance.csv", index=False)

print("="*80)
print("MODELO E RESULTADOS SALVOS")
print("="*80)

print(f"\n✓ lightgbm_model.txt: Modelo treinado")
print(f"✓ lightgbm_metrics.json: Métricas de desempenho")
print(f"✓ lightgbm_feature_importance.csv: Importância das features")
print(f"\nLocalização: {results_dir.absolute()}")

## Seção 6: Resumo e Conclusões

In [ ]:
print("\n" + "="*80)
print("RESUMO - MODELAGEM COM LIGHTGBM")
print("="*80)

print(f"""
1. MODELO TREINADO:
   ✓ Algoritmo: LightGBM (Light Gradient Boosting Machine)
   ✓ Tipo: Classificação binária
   ✓ Variáveis: {X_train.shape[1]}
   ✓ Amostras de treino: {X_train.shape[0]:,}
   ✓ Amostras de teste: {X_test.shape[0]:,}

2. DESEMPENHO NO TESTE:
   ✓ Acurácia: {metrics_test['Acurácia']:.4f} ({metrics_test['Acurácia']*100:.2f}%)
   ✓ Precisão: {metrics_test['Precisão']:.4f}
   ✓ Recall: {metrics_test['Recall']:.4f}
   ✓ F1-Score: {metrics_test['F1-Score']:.4f}
   ✓ AUC-ROC: {metrics_test['AUC-ROC']:.4f}

3. INTERPRETAÇÃO:
   - AUC-ROC mede a capacidade do modelo discriminar óbitos de sobrevivências
   - Valores próximos a 1.0 indicam excelente desempenho
   - Recall importante: Queremos identificar o máximo de óbitos possível

4. FEATURES MAIS IMPORTANTES:
   Top 3: {', '.join(feature_importance.head(3)['Feature'].tolist())}

5. PRÓXIMOS PASSOS:
   → Treinar XGBoost e CatBoost
   → Comparar desempenho dos três algoritmos
   → Análise SHAP para explicabilidade
   → Análise de justiça algorítmica
""")

print("="*80)
print("Modelagem com LightGBM concluída! Prossiga para o notebook 04_modeling_xgboost.ipynb")
print("="*80)